# trainer-subclass-extend — faded example 2: Three-level MRO chain accumulates keys via super()

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-subclass-extend`. The last cell reports your progress on the `Trainer: subclass extend pattern` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: subclass extend pattern` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`trainer-subclass-extend`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "trainer-subclass-extend"
DD_SUBTOPIC = "Trainer: subclass extend pattern"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When multiple subclasses form a chain (A inherits B inherits C), calling `super()._step(batch)` at each level walks Python's MRO in order. The final dict accumulates keys added at every level. Each subclass must call `super()._step(batch)` — not the grandparent directly — to respect the MRO.

## Faded exercise 2

A `BaseTrainer` is provided with `_step(batch)` returning `{'loss': float(batch[0].abs().sum())}`. `Level1Trainer(BaseTrainer)` calls `super()._step(batch)` and adds `'l1_key': 1`. Build `Level2Trainer(Level1Trainer)` whose `_step(batch)` calls `super()._step(batch)` (which walks through Level1 and then Base), then adds `'l2_key': 2` to the dict. The blank is the `super()._step(batch)` delegation in `Level2Trainer._step`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

class BaseTrainer:
    def __init__(self, model, lr=1e-2):
        self.model = model
        self.opt = t.optim.SGD(model.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

    def _step(self, batch):
        x, y = batch
        pred = self.model(x)
        loss = self.loss_fn(pred, y)
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        return {'loss': loss.item()}

class Level1Trainer(BaseTrainer):
    def _step(self, batch):
        d = super()._step(batch)
        d['l1_key'] = 1
        return d

class Level2Trainer(Level1Trainer):
    def _step(self, batch):
        raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
        d['l2_key'] = 2
        return d


def _test():
    import torch as t
    import torch.nn as nn
    t.manual_seed(1)
    model = nn.Linear(2, 1)
    trainer = Level2Trainer(model, lr=0.01)
    batch = (t.randn(3, 2), t.randn(3, 1))
    result = trainer._step(batch)
    assert 'loss' in result, 'base loss key missing'
    assert 'l1_key' in result, 'Level1 key missing'
    assert 'l2_key' in result, 'Level2 key missing'
    assert result['l1_key'] == 1
    assert result['l2_key'] == 2


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class BaseTrainer:
    def __init__(self, model, lr=1e-2):
        self.model = model
        self.opt = t.optim.SGD(model.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

    def _step(self, batch):
        x, y = batch
        pred = self.model(x)
        loss = self.loss_fn(pred, y)
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        return {'loss': loss.item()}

class Level1Trainer(BaseTrainer):
    def _step(self, batch):
        d = super()._step(batch)
        d['l1_key'] = 1
        return d

class Level2Trainer(Level1Trainer):
    def _step(self, batch):
        d = super()._step(batch)
        d['l2_key'] = 2
        return d
```
</details>